# PFAS Groundwater — V1: Inductive GraphSAGE-hetero vs HGT (T1a, head-to-head)

**Task T1a**: predict EPA-2024 regulatory PFAS exceedance (binary) from context features only  
(strict predictive mode — no PFAS measurement as a feature, no lat/lon, C-LOC.1).

**This notebook is AUTONOMOUS** (CLAUDE.md §4): it bootstraps `src/` and the versioned  
dataset via `git clone` (no Google Drive, no gdown), installs PyTorch Geometric for the  
Colab torch wheel, then runs the full V1 head-to-head via `V.run(smoke=...)`.

**V1 question:** does replacing the HGT encoder by an INDUCTIVE heterogeneous GraphSAGE  
(HeteroConv{relation: SAGEConv(mean)}, aggr='sum') with explicit regularisation  
(DropEdge p=0.2, GraphNorm, neighbour sampling fan-out=10) reduce spatial overfitting  
and/or change the verdict vs the in-run XGB-tabular wall?

**One protocol, two encoders:**
- arm `hgt`: HGTConv (reference, same as hgt_fusion_stacking_t1)
- arm `hetero_sage_v1`: inductive HeteroConv+SAGEConv with V1 regularisation

**Anti-leak design** (eval-validated, `experiments/v1_inductive_sage/EVAL_PROTOCOL_V1.md`):
- Spatial-block CV k=8; cross-block edges cut **per relation** + asserted to 0 (C-SPAT.2/5)
- Inductive: TRAIN-TRAIN edges only at fit time; test well aggregates from train neighbours (C-SPAT.4)
- 61 strict context features, no PFAS measurement, no lat/lon (C-LOC.1)
- Threshold from OOF/VAL probas only, never from test (C-THR)
- In-run XGB-tabular wall computed on the SAME 8 folds (apples-to-apples comparison)
- Paired Nadeau-Bengio + Wilcoxon tests; reality bar = 0.03 AUC above noise

**Training-curve output (CLAUDE.md §3.8):** `_plot_training_curves` in `src/v1_inductive_sage_t1.py`  
writes `training_curves_hgt.png` and `training_curves_hetero_sage_v1.png` to `experiments/v1_inductive_sage/`.  
The convergence diagnostic (`comparison.convergence`, `under_training_flag`) is printed and included in REPORT.md.

---
**CRITICAL — push the code first (see Cell 0 below).**  
`src/gnn_hetero_t1.py` (modified) and `src/v1_inductive_sage_t1.py` (new) are currently  
**untracked/modified** in git. Colab only sees committed + pushed files. You MUST commit  
and push both before opening this notebook on Colab, then set `GIT_REF` to that commit.

---
> `SMOKE_TEST=True`: CPU sanity check (~1-3 min, 500-well subsample, 3 blocks, 15 epochs).  
> `SMOKE_TEST=False`: full GPU run — two encoders x (spatial + random) = 4 CV passes + in-run XGB wall.

## Cell 0 — BEFORE YOU RUN: push the code to the remote

The following files are **not yet committed** (or are locally modified) in git:
- `src/gnn_hetero_t1.py` — modified to add `hetero_sage_v1` encoder, training-curve history, convergence diagnostic (§3.8)
- `src/v1_inductive_sage_t1.py` — new V1 runner
- `notebooks/v1_inductive_sage_colab.ipynb` — this notebook

**Colab only sees what is committed and pushed to `REPO_URL`.** Run the following  
locally before opening this notebook on Colab:

```bash
git add src/gnn_hetero_t1.py src/v1_inductive_sage_t1.py notebooks/v1_inductive_sage_colab.ipynb
git commit -m 'feat(v1): inductive hetero-SAGE vs HGT T1a, training-curve diagnostic (§3.8)'
git push
```

Then set `GIT_REF` in the parameter cell below to the branch (e.g. `"main"`) or the  
resulting commit SHA. **Do not hardcode a stale SHA** — use `"main"` unless you need  
to pin to a specific commit for reproducibility.

The anti-stale-code guard in Cell 3 will stop the notebook with an explicit error if the  
cloned code does not contain the required symbols.

In [ ]:
# ============================================================
# USER PARAMETERS — adjust before running
# ============================================================

SMOKE_TEST = False        # True = fast CPU sanity check (~1-3 min); False = full GPU run

# IMPORTANT: set GIT_REF to the branch/commit that contains:
#   src/gnn_hetero_t1.py (modified), src/v1_inductive_sage_t1.py (new)
# See the markdown cell above for the exact git commands.
REPO_URL = "https://github.com/dnwiloic/pfas-gnn.git"
GIT_REF  = "main"        # branch name or full commit SHA — must include both src files
DATA_PATH = "data/CA-PFAS-ASGWS.parquet"  # relative to repo root (versioned in repo)

EXP_DIR = "experiments/v1_inductive_sage"   # artefacts written here (workspace-relative)
SMOKE_EXP_DIR = "experiments/v1_inductive_sage/smoke"

# Full-run V1 hyperparameters (match src/v1_inductive_sage_t1.py defaults — only change for ablation)
FULL_N_BLOCKS    = 8      # spatial KMeans CV blocks
FULL_HIDDEN      = 64     # hidden dim (both encoders)
FULL_LAYERS      = 2
FULL_DROPOUT     = 0.3
FULL_HEADS       = 4      # HGT attention heads (hetero-SAGE ignores this)
FULL_MAX_EPOCHS  = 400
FULL_PATIENCE    = 50
FULL_LR          = 5e-3
FULL_WEIGHT_DECAY = 5e-4
COMPUTE_DELTA    = True   # also run random-block CV to measure delta(random - spatial)

SEED = 42

# ============================================================
# DURATION ESTIMATE (SMOKE_TEST=False, Colab T4 GPU)
# ============================================================
# V1 runs: 2 encoders x (spatial + random) = 4 CV passes, each with 8 folds.
# Plus in-run XGB-tabular wall computed per fold inside the spatial pass.
#
# Prior measurement (hgt_fusion_stacking_t1, Colab T4 GPU):
#   HGT standalone 8-fold spatial pass ~ 17.5 min
# hetero-SAGE is lighter (no attention), so V1 = ~2 x (HGT + SAGE) x 2 regimes
# Conservative estimate: ~50-70 min total on Colab T4 GPU.
#
# Checkpointing: metrics.json written after EACH encoder's spatial pass
# (incremental, so a disconnect never loses work already done).
# ============================================================

print("Parameters set.")
print(f"  SMOKE_TEST : {SMOKE_TEST}")
print(f"  REPO_URL   : {REPO_URL}")
print(f"  GIT_REF    : {GIT_REF}")
print(f"  DATA_PATH  : {DATA_PATH}")
print(f"  EXP_DIR    : {EXP_DIR}")
if not SMOKE_TEST:
    print()
    print("  Full-run estimate (Colab T4 GPU): ~50-70 min")
    print("  Checkpoint: metrics.json written after each encoder's spatial CV pass.")

## Cell 1 — GPU detection & runtime versions

**Note:** the XGBoost in-run wall runs on CPU (tree_method='hist'); only the GNN training  
is GPU-accelerated. For `SMOKE_TEST=True` a CPU runtime (High-RAM) is fine and avoids  
GPU quota warnings. For the full run, use **T4 GPU** (Runtime > Change runtime type).

In [ ]:
import sys, platform, subprocess

print(f"Python  : {sys.version.split()[0]}")
print(f"Platform: {platform.platform()}")

IN_COLAB = False
try:
    import google.colab  # noqa
    IN_COLAB = True
except ImportError:
    pass
print(f"IN_COLAB: {IN_COLAB}")

try:
    import torch
    cuda_ok = torch.cuda.is_available()
    print(f"torch   : {torch.__version__}  CUDA avail: {cuda_ok}")
    if cuda_ok:
        print(f"GPU     : {torch.cuda.get_device_name(0)}")
        print(f"CUDA    : {torch.version.cuda}")
    else:
        print("WARNING: no GPU detected.")
        if SMOKE_TEST:
            print("  SMOKE_TEST=True -> CPU run is expected and fine.")
        else:
            print("  SMOKE_TEST=False -> full run needs GPU (~50-70 min on T4).")
            print("  Go to: Runtime > Change runtime type > T4 GPU")
except ImportError:
    print("torch not yet installed (will be available after Cell 2 installs).")

print()
print("Note: XGBoost in-run wall uses CPU (tree_method='hist') — no GPU needed for it.")

## Cell 2 — Install pinned dependencies + verify imports

PyG compiled extensions must match `torch.__version__` + CUDA tag.  
We detect both at runtime and install from the matching wheel index.  
This cell is idempotent: if PyG is already importable it skips the heavy install.

**V1-specific imports verified:** `dropout_edge`, `GraphNorm`, `SAGEConv`, `HeteroConv`  
(all required by `src/gnn_hetero_t1.py` for the `hetero_sage_v1` encoder).

In [ ]:
import subprocess, sys

# --- PyTorch Geometric (must match runtime torch + CUDA) ---
def ensure_pyg():
    try:
        import torch_geometric
        print(f"torch_geometric already present: {torch_geometric.__version__}")
        return
    except ImportError:
        pass
    import torch
    tv = torch.__version__.split("+")[0]
    cuda = torch.version.cuda
    tag = f"cu{cuda.replace('.', '')}" if (cuda and torch.cuda.is_available()) else "cpu"
    idx = f"https://data.pyg.org/whl/torch-{tv}+{tag}.html"
    print(f"Installing torch_geometric for torch={tv}, device={tag}")
    print(f"  Wheel index: {idx}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "torch_geometric", "-f", idx], check=True)
    # Optional compiled helpers (scatter/sparse) — ignore failure on CPU
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "torch_scatter", "torch_sparse", "-f", idx])

ensure_pyg()

# --- V1-specific import verification ---
# These symbols are ALL required by gnn_hetero_t1.build_model_t1 for hetero_sage_v1:
import torch_geometric
print(f"PyG: {torch_geometric.__version__}")

from torch_geometric.utils import dropout_edge      # V1 DropEdge regularisation
print("  dropout_edge import OK")
from torch_geometric.nn import GraphNorm            # V1 GraphNorm (replaces LayerNorm)
print("  GraphNorm import OK")
from torch_geometric.nn import SAGEConv             # V1 inductive message passing
print("  SAGEConv import OK")
from torch_geometric.nn import HeteroConv           # V1 per-relation routing
print("  HeteroConv import OK")
from torch_geometric.nn import HGTConv              # reference encoder
print("  HGTConv import OK")
print()
print("All V1 PyG symbols verified.")

# --- XGBoost (in-run wall) ---
try:
    import xgboost as xgb
    print(f"xgboost : {xgb.__version__}")
except ImportError:
    print("xgboost not found — installing...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "xgboost>=2.0"],
                   check=True)
    import xgboost as xgb
    print(f"xgboost : {xgb.__version__} (just installed)")

# --- Other deps ---
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "pyarrow>=14.0", "pyyaml>=6.0", "scikit-learn>=1.4",
                "matplotlib>=3.7"], check=True)

print("All dependencies satisfied.")

## Cell 3 — Clone repo (brings src/ AND data/ — no Drive, no gdown)

The dataset `data/CA-PFAS-ASGWS.parquet` is **versioned in the repo** and arrives  
automatically with the clone. This cell is idempotent: if the repo dir already  
exists it only checks out the requested ref.

**Anti-stale-code guard:** after bootstrap, verifies that the cloned code at `GIT_REF`  
actually contains the V1 symbols (`run`, `_plot_training_curves`, `_convergence_diag`,  
`hetero_sage_v1`). If any are missing, the notebook stops with an explicit message  
telling you to push the missing files and update `GIT_REF`.

In [ ]:
import os, sys, pathlib

REPO_DIR = "/content/pfas-gnn" if IN_COLAB else os.getcwd()

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        print(f"Cloning {REPO_URL} into {REPO_DIR} ...")
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    print(f"Checking out {GIT_REF} ...")
    subprocess.run(["git", "-C", REPO_DIR, "checkout", GIT_REF], check=True)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f"workdir : {os.getcwd()}")

# ---- Anti-stale-code guard: V1 runner ----
# If this fails: git add src/gnn_hetero_t1.py src/v1_inductive_sage_t1.py
#                git commit -m '...' && git push
# Then update GIT_REF to point to that commit and re-run this cell.
V1_MODULE_PATH = pathlib.Path(REPO_DIR) / "src" / "v1_inductive_sage_t1.py"
assert V1_MODULE_PATH.exists(), (
    f"FATAL: src/v1_inductive_sage_t1.py not found in the cloned repo at GIT_REF={GIT_REF!r}.\n"
    "You must commit and push the V1 module BEFORE running this notebook on Colab:\n"
    "  git add src/gnn_hetero_t1.py src/v1_inductive_sage_t1.py notebooks/v1_inductive_sage_colab.ipynb\n"
    "  git commit -m 'feat(v1): inductive hetero-SAGE vs HGT T1a'\n"
    "  git push\n"
    "Then set GIT_REF to the resulting branch or commit SHA and re-run this cell."
)

# Verify V1 runner symbols
_v1_src = V1_MODULE_PATH.read_text()
for symbol in ["def run(", "def _plot_training_curves(", "def _convergence_diag(",
               "hetero_sage_v1", "ENCODERS"]:
    assert symbol in _v1_src, (
        f"Symbol {symbol!r} not found in src/v1_inductive_sage_t1.py — "
        f"the repo at GIT_REF={GIT_REF!r} may point to an incomplete commit.\n"
        "Push the complete module (including §3.8 patch) and update GIT_REF."
    )
    print(f"  v1_inductive_sage_t1 : {symbol!r:<45} OK")

# Verify gnn_hetero_t1 has the V1 encoder patch (hetero_sage_v1 + history + dropout_edge)
HETERO_MODULE_PATH = pathlib.Path(REPO_DIR) / "src" / "gnn_hetero_t1.py"
assert HETERO_MODULE_PATH.exists(), (
    f"FATAL: src/gnn_hetero_t1.py not found at GIT_REF={GIT_REF!r}.\n"
    "Commit and push as shown above."
)
_hetero_src = HETERO_MODULE_PATH.read_text()
for symbol in ["hetero_sage_v1", "dropout_edge", "GraphNorm", "history_val_auc", "train_diag"]:
    assert symbol in _hetero_src, (
        f"Symbol {symbol!r} not found in src/gnn_hetero_t1.py — "
        "the §3.8 patch (training-curve history + convergence diagnostic) is missing.\n"
        f"Push the patched gnn_hetero_t1.py at GIT_REF={GIT_REF!r}."
    )
    print(f"  gnn_hetero_t1        : {symbol!r:<45} OK")

# Verify dataset present
assert os.path.exists(DATA_PATH), (
    f"FATAL: dataset missing at {DATA_PATH}.\n"
    "The parquet file should be versioned in the repo at data/CA-PFAS-ASGWS.parquet.\n"
    "Check REPO_URL and GIT_REF — the clone may have failed or the file is not committed."
)
print(f"\ndataset : {DATA_PATH}  ({os.path.getsize(DATA_PATH) // 1024} KB)")
print("Code + dataset guard PASSED.")

## Cell 4 — Load dataset & integrity check

Expected shape: **46338 rows x 201 columns**, **11333 unique wells**.  
Required columns: `gm_well_id`, `latitude`, `longitude`, `PFOA_ngL`.  
Any mismatch triggers an explicit stop — no silent failure downstream.

In [ ]:
import pandas as pd

EXPECTED_ROWS  = 46338
EXPECTED_COLS  = 201
EXPECTED_WELLS = 11333
KEY_COLS = ["gm_well_id", "latitude", "longitude", "PFOA_ngL"]

df_probe = pd.read_parquet(DATA_PATH)
n_rows, n_cols = df_probe.shape
n_wells = df_probe["gm_well_id"].nunique()
missing_keys = [c for c in KEY_COLS if c not in df_probe.columns]

print(f"Shape  : {n_rows} rows x {n_cols} cols")
print(f"Wells  : {n_wells} unique")
print(f"Memory : {df_probe.memory_usage(deep=True).sum() // 1024 // 1024} MB")

if missing_keys:
    raise RuntimeError(
        f"FATAL: key columns missing from dataset: {missing_keys}.\n"
        f"Check DATA_PATH={DATA_PATH!r} and GIT_REF={GIT_REF!r}.\n"
        "The parquet must contain: " + ", ".join(KEY_COLS)
    )
print(f"Key columns {KEY_COLS} — all present")

if not SMOKE_TEST:
    if n_rows != EXPECTED_ROWS:
        raise RuntimeError(
            f"FATAL: row count mismatch — got {n_rows}, expected {EXPECTED_ROWS}.\n"
            f"Clone may be stale or DATA_PATH={DATA_PATH!r} points to the wrong file.\n"
            f"Check GIT_REF={GIT_REF!r}."
        )
    if n_cols != EXPECTED_COLS:
        raise RuntimeError(
            f"FATAL: column count mismatch — got {n_cols}, expected {EXPECTED_COLS}.\n"
            f"Check GIT_REF={GIT_REF!r}."
        )
    if n_wells != EXPECTED_WELLS:
        raise RuntimeError(
            f"FATAL: well count mismatch — got {n_wells}, expected {EXPECTED_WELLS}.\n"
            f"Check GIT_REF={GIT_REF!r} and DATA_PATH={DATA_PATH!r}."
        )
    print(f"Integrity check PASSED: {n_rows} x {n_cols}, {n_wells} wells.")
else:
    print(f"SMOKE_TEST=True — shape check relaxed (got {n_rows} x {n_cols}, {n_wells} wells).")
    print("  The module will subsample to 500 wells internally.")

del df_probe  # free memory; the module reloads via src.data.load()

## Cell 5 — Smoke-test (`SMOKE_TEST=True` only)

Calls `V.run(smoke=True)` which:
- subsamples 500 wells, 3 spatial blocks, 15 epochs (patience 6),
- runs **both encoders** (hgt + hetero_sage_v1) through the full V1 pipeline,
- computes the in-run XGB wall on the same folds,
- asserts cross-block edges = 0 per encoder,
- writes training-curve PNGs and metrics to `experiments/v1_inductive_sage/smoke/`.

When `SMOKE_TEST=False` this cell is skipped.

In [ ]:
import time, json, math
from pathlib import Path

if SMOKE_TEST:
    print("=" * 65)
    print("SMOKE-TEST: 500 wells / 3 blocks / 15 epochs — CPU run")
    print("Both encoders: hgt + hetero_sage_v1")
    print("=" * 65)

    from src import v1_inductive_sage_t1 as V

    t0 = time.time()
    smoke_out = V.run(
        smoke=True,
        write=True,
        exp_dir=SMOKE_EXP_DIR,
        seed=SEED,
        verbose=True,
    )
    elapsed_smoke = time.time() - t0

    # ---- end-to-end assertions ----
    # 1. Both encoders present
    enc = smoke_out.get("encoders", {})
    for enc_name in ("hgt", "hetero_sage_v1"):
        assert enc_name in enc, (
            f"SMOKE FAIL: encoder '{enc_name}' missing from output. "
            "Check that ENCODERS = ('hgt', 'hetero_sage_v1') in v1_inductive_sage_t1.py."
        )
        sp = enc[enc_name]["spatial"]

        # 2. Zero cross-block edges per encoder (C-SPAT.2/5)
        assert sp["n_cross_block_total"] == 0, (
            f"LEAK DETECTED [{enc_name}]: {sp['n_cross_block_total']} cross-block edges remain! "
            "Spatial-leakage violation C-SPAT.2/5."
        )
        print(f"  [{enc_name}] n_cross_block_total = 0  OK")

        # 3. Finite AUC
        auc = sp["metrics"]["roc_auc"]
        assert math.isfinite(auc), f"SMOKE FAIL [{enc_name}]: AUC is not finite ({auc})"
        print(f"  [{enc_name}] spatial AUC = {auc:.4f}  (smoke subset — not representative)")

    # 4. comparison.convergence present (§3.8)
    cmp = smoke_out.get("comparison", {})
    assert "convergence" in cmp, (
        "SMOKE FAIL: 'comparison.convergence' missing from output. "
        "The §3.8 convergence diagnostic was not written."
    )
    print("  comparison.convergence present  OK")

    # 5. Training-curve PNGs written
    smoke_dir = Path(SMOKE_EXP_DIR)
    for enc_name in ("hgt", "hetero_sage_v1"):
        png = smoke_dir / f"training_curves_{enc_name}.png"
        assert png.exists(), (
            f"SMOKE FAIL: training_curves_{enc_name}.png not written to {SMOKE_EXP_DIR}/. "
            "Check that matplotlib is installed and _plot_training_curves ran."
        )
        print(f"  training_curves_{enc_name}.png written  OK")

    # 6. metrics.json written
    assert (smoke_dir / "metrics.json").exists(), (
        f"SMOKE FAIL: metrics.json not written to {SMOKE_EXP_DIR}/"
    )
    print("  metrics.json written  OK")

    print()
    print(f"Smoke test elapsed: {elapsed_smoke:.0f}s ({elapsed_smoke/60:.1f} min)")

    # ---- full-run duration estimate (extrapolation) ----
    # Smoke: 500 wells, 3 blocks, 15 epochs, 1 regime (spatial only in run())
    # Full:  ~11333 wells, 8 blocks, 400 epochs, 2 regimes (spatial + random)
    # Both encoders run in both cases, so encoder count cancels.
    scale_epochs = 400 / 15
    scale_blocks = 8 / 3
    scale_regime = 2      # spatial + random Δ arm
    # GNN scales sublinearly with nodes due to neighbour sampling; use 0.5 exponent
    scale_wells = (11333 / 500) ** 0.5
    est_cpu_s = elapsed_smoke * scale_epochs * scale_blocks * scale_regime * scale_wells
    est_gpu_s = est_cpu_s / 12   # T4 GPU ~12x CPU speedup (conservative)
    print(f"Full-run wall-time estimate (CPU)   : {est_cpu_s/60:.0f} min ({est_cpu_s/3600:.1f} h)")
    print(f"Full-run wall-time estimate (T4 GPU): {est_gpu_s/60:.0f} min ({est_gpu_s/3600:.1f} h)")
    if est_gpu_s > 90 * 60:
        print()
        print("WARNING: estimated GPU time > 90 min. Checkpointing writes metrics.json")
        print("  after each encoder's spatial pass — a disconnect won't lose prior work.")
        if est_gpu_s > 4 * 3600:
            print("WARNING: estimated GPU time > 4 h. Consider COMPUTE_DELTA=False to halve budget.")
    else:
        print(f"  -> Fits comfortably in a Colab T4 session (~50-70 min expected).")

    print()
    print("SMOKE-TEST PASSED. Set SMOKE_TEST=False and switch to a GPU runtime for the full run.")

else:
    print("SMOKE_TEST=False — smoke cell skipped. Proceed to Cell 6 for the full GPU run.")

## Cell 6 — Full run (`SMOKE_TEST=False`)

Calls `V.run(smoke=False)` which runs:
1. **Spatial CV (8 blocks)** for both encoders (HGT then hetero-SAGE-v1),  
   plus the in-run XGB-tabular wall on the same folds.
2. **Random CV (8 blocks)** for both encoders, for the Δ(random − spatial) diagnostic.
3. **Paired tests** (Nadeau-Bengio + Wilcoxon) for SAGE vs HGT, SAGE vs wall, HGT vs wall.
4. **Training-curve PNGs** written to `experiments/v1_inductive_sage/`.
5. **Convergence diagnostic** (`under_training_flag`) per encoder.

**Checkpoint:** `metrics.json` is written incrementally after **each encoder's spatial pass**,  
so a Colab disconnect never loses work already completed.

All artefacts go to `experiments/v1_inductive_sage/` inside the cloned workspace.

In [ ]:
import time
from pathlib import Path

if not SMOKE_TEST:
    print("=" * 65)
    print("FULL RUN: V1 head-to-head HGT vs hetero-SAGE-v1, T1a")
    print("=" * 65)
    print(f"  encoders    : hgt + hetero_sage_v1")
    print(f"  hidden={FULL_HIDDEN}  layers={FULL_LAYERS}  heads={FULL_HEADS}  dropout={FULL_DROPOUT}")
    print(f"  max_epochs={FULL_MAX_EPOCHS}  patience={FULL_PATIENCE}  lr={FULL_LR}  wd={FULL_WEIGHT_DECAY}")
    print(f"  n_blocks={FULL_N_BLOCKS}  compute_delta={COMPUTE_DELTA}  seed={SEED}")
    print(f"  V1 regularisation: drop_edge=0.2, GraphNorm=True, neighbor_fanout=10")
    print()
    print("Checkpoint: metrics.json written after each encoder's spatial CV pass.")
    print("Training-curve PNGs written after all encoders complete (in write block).")
    print()

    from src import v1_inductive_sage_t1 as V

    t0 = time.time()
    full_out = V.run(
        smoke=False,
        n_blocks=FULL_N_BLOCKS,
        hidden=FULL_HIDDEN,
        layers=FULL_LAYERS,
        dropout=FULL_DROPOUT,
        heads=FULL_HEADS,
        max_epochs=FULL_MAX_EPOCHS,
        patience=FULL_PATIENCE,
        lr=FULL_LR,
        weight_decay=FULL_WEIGHT_DECAY,
        compute_delta=COMPUTE_DELTA,
        write=True,
        exp_dir=EXP_DIR,
        seed=SEED,
        verbose=True,
    )
    elapsed_full = time.time() - t0

    # Verify anti-leak assertion held for the full run
    for enc_name in ("hgt", "hetero_sage_v1"):
        xb = full_out["encoders"][enc_name]["spatial"]["n_cross_block_total"]
        assert xb == 0, (
            f"LEAK DETECTED [{enc_name}] in full run: {xb} cross-block edges remain. "
            "DO NOT use these results — spatial leakage violates C-SPAT.2/5."
        )

    print(f"\nFull run completed in {elapsed_full:.0f}s ({elapsed_full/60:.1f} min)")
    print("Cross-block edges: 0 for both encoders  OK")
    print(f"Artefacts in: {EXP_DIR}/")

else:
    full_out = None
    print("SMOKE_TEST=True — full run cell skipped.")

## Cell 7 — Display results

Reads `experiments/v1_inductive_sage/metrics.json` and renders:
1. **HGT vs hetero-SAGE-v1 comparison table** (AUC spatial/random, Δ, overfit gap)
2. **Paired-test verdict vs the in-run wall** (Nadeau-Bengio + Wilcoxon)
3. **Convergence / under-training diagnostic** — **AVERTISSEMENT** if `under_training_flag=YES`
4. **Training-curve PNG figures** displayed inline (both encoders)
5. Full `REPORT.md` text

Reading from disk ensures the display matches what was persisted (safe even if `full_out`  
is not in memory after a reconnect).

In [ ]:
import json, math
from pathlib import Path
import numpy as np

# Load from disk (works even if full_out is not in scope)
_exp_root = Path(EXP_DIR) if not SMOKE_TEST else Path(SMOKE_EXP_DIR)
metrics_path = _exp_root / "metrics.json"
report_path  = _exp_root / "REPORT.md"

if not metrics_path.exists():
    print(f"metrics.json not found at {metrics_path}.")
    print("Run Cell 5 (smoke) or Cell 6 (full run) first.")
else:
    out = json.loads(metrics_path.read_text())
    meta = out["meta"]
    E    = out["encoders"]
    cmp  = out.get("comparison", {})
    wall = out.get("in_run_xgb_wall_spatial", {})

    print(f"smoke={meta['smoke']}  seed={meta['seed']}  blocks={meta['n_blocks']}  "
          f"features={meta['n_features']}  elapsed={meta.get('elapsed_s', float('nan')):.0f}s")
    print()

    # ---- 1. Main comparison table: HGT vs hetero-SAGE vs XGB wall ----
    print("=" * 75)
    print("TABLE 1 — Spatial-block OOF results (row-level)")
    print("=" * 75)
    hdr = (f"{'encoder':<28} {'AUC OOF':>8}  {'95% CI':>15}  "
           f"{'AUC pfm':>8}  {'F1':>6}  {'PR-AUC':>7}  {'Brier':>7}  {'ECE':>6}")
    print(hdr)
    print("-" * len(hdr))

    def _row(label, sp, ci_dict):
        m = sp["metrics"]
        pfm = float(np.nanmean(sp["per_fold_auc"])) if sp.get("per_fold_auc") else float("nan")
        ci_lo = ci_dict.get("ci_low", float("nan"))
        ci_hi = ci_dict.get("ci_high", float("nan"))
        return (f"{label:<28} {m['roc_auc']:>8.4f}  "
                f"[{ci_lo:.3f},{ci_hi:.3f}]  "
                f"{pfm:>8.4f}  {m['f1']:>6.4f}  {m['pr_auc']:>7.4f}  "
                f"{m['brier']:>7.4f}  {m.get('ece', float('nan')):>6.4f}")

    if "hgt" in E:
        sp_hgt = E["hgt"]["spatial"]
        print(_row("HGT (reference)", sp_hgt, sp_hgt.get("auc_ci95", {})))
    if "hetero_sage_v1" in E:
        sp_sage = E["hetero_sage_v1"]["spatial"]
        print(_row("hetero-SAGE-v1 (inductive)", sp_sage, sp_sage.get("auc_ci95", {})))
    if wall:
        print(_row("XGB-tabular (in-run wall)", wall, wall.get("auc_ci95", {})))

    # ---- 2. Overfitting diagnostic ----
    print()
    print("=" * 75)
    print("TABLE 2 — Overfitting diagnostic (fit vs val AUC at best epoch, mean over folds)")
    print("=" * 75)
    hdr2 = (f"{'encoder':<28} {'fit AUC':>8}  {'val AUC':>8}  "
            f"{'fit-val gap':>11}  {'F1 gap':>8}  {'best epoch':>10}")
    print(hdr2)
    print("-" * len(hdr2))
    for label, key in [("HGT", "hgt"), ("hetero-SAGE-v1", "hetero_sage_v1")]:
        if key not in E:
            continue
        o = E[key]["spatial"].get("overfit_diag", {})
        print(f"{label:<28} {o.get('fit_auc_mean', float('nan')):>8.4f}  "
              f"{o.get('val_auc_mean', float('nan')):>8.4f}  "
              f"{o.get('gap_auc_mean', float('nan')):>+11.4f}  "
              f"{o.get('gap_f1_mean', float('nan')):>+8.4f}  "
              f"{o.get('best_epoch_mean', float('nan')):>10.1f}")
    og = cmp.get("overfit_gap_auc", {})
    red = og.get("reduction_sage_vs_hgt", float("nan"))
    if math.isfinite(red):
        print(f"\nOverfit gap reduction (HGT gap - SAGE gap) = {red:+.4f} AUC")
        print("  Positive = SAGE generalises with a smaller fit-to-val gap (V1 motivation).")

    # ---- 3. Paired-test verdicts vs in-run wall ----
    print()
    print("=" * 75)
    print("TABLE 3 — Paired tests vs in-run XGB wall + head-to-head (8 spatial folds)")
    print("=" * 75)
    wall_mean = cmp.get("wall_in_run_auc_mean", float("nan"))
    wall_oof  = cmp.get("wall_in_run_auc_oof_global", float("nan"))
    print(f"In-run XGB wall: OOF-global={wall_oof:.4f}  per-fold-mean={wall_mean:.4f}")
    print()
    hdr3 = f"{'comparison':<30} {'gain':>7}  {'NB p':>8}  {'Wilc p':>8}  {'verdict'}"
    print(hdr3)
    print("-" * 70)
    for label, key in [("HGT vs wall", "hgt_vs_wall"),
                       ("hetero-SAGE-v1 vs wall", "hetero_sage_v1_vs_wall")]:
        rec = cmp.get(key, {})
        nb_p = rec.get("paired_vs_wall", {}).get("nadeau_bengio", {}).get("p", float("nan"))
        wc_p = rec.get("paired_vs_wall", {}).get("wilcoxon", {}).get("p", float("nan"))
        gain = rec.get("gain_vs_in_run_wall", float("nan"))
        verdict = rec.get("verdict", "n/a")
        print(f"{label:<30} {gain:>+7.4f}  {nb_p:>8.4f}  {wc_p:>8.4f}  {verdict}")
    sh = cmp.get("hetero_sage_v1_vs_hgt", {})
    sh_nb = sh.get("paired", {}).get("nadeau_bengio", {}).get("p", float("nan"))
    sh_wc = sh.get("paired", {}).get("wilcoxon", {}).get("p", float("nan"))
    sh_d  = sh.get("per_fold_mean_diff", float("nan"))
    print(f"{'hetero-SAGE-v1 vs HGT':<30} {sh_d:>+7.4f}  {sh_nb:>8.4f}  {sh_wc:>8.4f}  "
          f"{'differs' if (math.isfinite(sh_nb) and sh_nb < 0.05) else 'no_robust_diff'}")
    print()
    print("Reality rule (eval C-CMP): robust gain requires p<0.05 AND >0.03 AUC above noise.")

    # ---- 4. Delta (random - spatial) ----
    if COMPUTE_DELTA and "hetero_sage_v1" in E and "random" in E["hetero_sage_v1"]:
        print()
        print("=" * 75)
        print("TABLE 4 — Delta(random - spatial) per encoder (C-SPAT.6)")
        print("=" * 75)
        hdr4 = f"{'encoder':<28} {'spatial AUC':>12}  {'random AUC':>12}  {'delta':>8}"
        print(hdr4)
        print("-" * len(hdr4))
        for label, key in [("HGT", "hgt"), ("hetero-SAGE-v1", "hetero_sage_v1")]:
            if key not in E:
                continue
            sp_auc  = E[key]["spatial"]["metrics"]["roc_auc"]
            rnd_auc = E[key].get("random", {}).get("metrics", {}).get("roc_auc", float("nan"))
            delta   = E[key].get("delta_random_minus_spatial", float("nan"))
            print(f"{label:<28} {sp_auc:>12.4f}  {rnd_auc:>12.4f}  {delta:>+8.4f}")

    # ---- 5. Convergence / under-training diagnostic (§3.8) ----
    conv = cmp.get("convergence", {})
    if conv:
        print()
        print("=" * 75)
        print("TABLE 5 — Convergence / under-training diagnostic (CLAUDE.md §3.8)")
        print("=" * 75)
        hdr5 = (f"{'encoder':<28} {'epochs mean/min':>16}  {'early-stop':>10}  "
                f"{'best-last-20%':>13}  {'under-training'}")
        print(hdr5)
        print("-" * len(hdr5))
        any_undertraining = False
        for label, key in [("HGT", "hgt"), ("hetero-SAGE-v1", "hetero_sage_v1")]:
            c = conv.get(key, {})
            flag = c.get("under_training_flag", False)
            if flag:
                any_undertraining = True
            flag_str = "YES  (***)" if flag else "no"
            print(f"{label:<28} "
                  f"{c.get('n_epochs_ran_mean', float('nan')):>7.1f} / {c.get('n_epochs_ran_min', 0):>5}  "
                  f"{c.get('frac_folds_early_stopped', float('nan')):>10.0%}  "
                  f"{c.get('frac_folds_best_in_last_20pct', float('nan')):>13.0%}  "
                  f"{flag_str}")
        if any_undertraining:
            print()
            print("*** AVERTISSEMENT under_training_flag = YES ***")
            print("For at least one encoder, the best validation epoch fell in the last")
            print("20% of the run for >=50% of folds — a signature of premature early-stop.")
            print("ACTION REQUIRED: raise max_epochs (e.g. 600) and/or patience (e.g. 75)")
            print("and RELAUNCH BEFORE endorsing the AUC figures above.")
            print("See experiments/v1_inductive_sage/training_curves_<encoder>.png")
        else:
            print()
            print("under_training_flag = no for all encoders — early-stop appears stable.")

    # ---- 6. Display training-curve PNGs inline ----
    print()
    print("=" * 75)
    print("Training-curve figures (CLAUDE.md §3.8)")
    print("=" * 75)
    try:
        import matplotlib.pyplot as plt
        import matplotlib.image as mpimg
        for enc_name in ("hgt", "hetero_sage_v1"):
            png_path = _exp_root / f"training_curves_{enc_name}.png"
            if png_path.exists():
                img = mpimg.imread(str(png_path))
                fig, ax = plt.subplots(1, 1, figsize=(12, 4))
                ax.imshow(img)
                ax.axis("off")
                ax.set_title(f"training_curves_{enc_name}.png", fontsize=10)
                plt.tight_layout()
                plt.show()
                print(f"  Displayed: {png_path}")
            else:
                print(f"  training_curves_{enc_name}.png not found (run Cell 6 first).")
    except Exception as e:
        print(f"  Could not display PNGs inline: {e}")
        print(f"  Find them at: {_exp_root}/training_curves_*.png")

    # ---- 7. REPORT.md ----
    if report_path.exists():
        print()
        print("=" * 75)
        print("REPORT.md")
        print("=" * 75)
        print(report_path.read_text())

## Cell 8 — Persist outputs

> **WARNING: the Colab workspace is EPHEMERAL.** All files in `/content/pfas-gnn/`
> (including training-curve PNGs, metrics, and REPORT.md) are **permanently lost**
> when the runtime disconnects or times out. Without running this cell your results
> and the §3.8 training-curve figures will be gone.

**Option A (quick):** `files.download()` creates a local zip archive on your machine.

**Option B (recommended):** `git commit + push` writes results back to the repo so  
they survive, are versioned, and are visible to subsequent agents.  
Requires push access to `REPO_URL` (configure git credentials first).

Run **at least one option** before closing the session.

In [ ]:
import shutil, os
from pathlib import Path

_persist_dir = Path(EXP_DIR) if not SMOKE_TEST else Path(SMOKE_EXP_DIR)

if not _persist_dir.exists():
    print(f"No outputs found at {_persist_dir}. Run Cell 5 or Cell 6 first.")
else:
    files_found = [p for p in sorted(_persist_dir.rglob("*")) if p.is_file()]
    print(f"Outputs in {_persist_dir}/ ({len(files_found)} files):")
    for p in files_found:
        print(f"  {p.relative_to(_persist_dir)}  ({p.stat().st_size // 1024} KB)")

    # ==== Option A: download zip archive (includes PNGs) ====
    # shutil.make_archive uses root_dir + base_dir so the zip contains
    # `v1_inductive_sage/metrics.json`, `v1_inductive_sage/training_curves_hgt.png`, etc.
    archive_stem = "v1_inductive_sage_outputs"
    archive_path = shutil.make_archive(
        archive_stem, "zip",
        root_dir=str(_persist_dir.parent),
        base_dir=_persist_dir.name
    )
    archive_size = os.path.getsize(archive_path) // 1024
    print(f"\nOption A — Archive: {archive_path} ({archive_size} KB)")
    print("  Contains: metrics.json, REPORT.md, config.yaml,")
    print("            training_curves_hgt.png, training_curves_hetero_sage_v1.png")
    if IN_COLAB:
        from google.colab import files
        files.download(archive_path)
        print("  Download triggered — save this zip to your local machine.")
    else:
        print(f"  (Not in Colab — archive at {os.path.abspath(archive_path)})")

    # ==== Option B: git commit + push ====
    # Uncomment and fill in your git identity if needed.
    # This pushes the experiment outputs back to REPO_URL so they are versioned.
    #
    # import subprocess
    # # (Optional) set identity if not configured:
    # subprocess.run(["git", "config", "user.email", "dnjomouloic@gmail.com"], cwd=REPO_DIR)
    # subprocess.run(["git", "config", "user.name", "Loic"], cwd=REPO_DIR)
    # _smoke_flag = "smoke" if SMOKE_TEST else "full"
    # subprocess.run(["git", "add",
    #     "experiments/v1_inductive_sage/metrics.json",
    #     "experiments/v1_inductive_sage/REPORT.md",
    #     "experiments/v1_inductive_sage/config.yaml",
    #     "experiments/v1_inductive_sage/training_curves_hgt.png",
    #     "experiments/v1_inductive_sage/training_curves_hetero_sage_v1.png",
    # ], cwd=REPO_DIR, check=True)
    # subprocess.run(["git", "commit", "-m",
    #     f"results(v1_inductive_sage): {_smoke_flag} run — HGT vs hetero-SAGE T1a from Colab GPU"
    # ], cwd=REPO_DIR, check=True)
    # subprocess.run(["git", "push"], cwd=REPO_DIR, check=True)
    # print("Option B — Results committed and pushed to", REPO_URL)

    print()
    print("REMINDER: without Option A download OR Option B git push, ALL outputs")
    print("(including training_curves_*.png and metrics.json) are permanently lost")
    print("when this Colab runtime disconnects or times out.")